# Rule V2 Regime Viewer

This notebook loads the local rule_v2 market basket, computes daily regime labels, and plots normalized index lines with shaded regime zones.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'research').exists() and (REPO_ROOT.parent / 'research').exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'research').exists():
    raise RuntimeError("Could not locate repo root containing research/")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from research.common.regime_viewer import build_rule_v2_regime_artifacts

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
START_DATE = '2022-07-22'
END_DATE = None  # use latest common local date when None
DATA_ROOT = REPO_ROOT / 'data/market'

artifacts = build_rule_v2_regime_artifacts(
    data_root=DATA_ROOT,
    start_date=START_DATE,
    end_date=END_DATE,
)

prices = artifacts.prices
norm = artifacts.normalized_prices
regimes = artifacts.regimes

print('Price window:', prices.index.min().date(), '->', prices.index.max().date())
print('Regime rows:', len(regimes))
display(regimes.tail())


In [ ]:
label_colors = {
    'risk_on': '#d9f2d9',
    'neutral': '#f2ecd9',
    'risk_off': '#f7d9d9',
}

fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True, height_ratios=[2.2, 1.0])

ax = axes[0]
for label in ['risk_on', 'neutral', 'risk_off']:
    mask = regimes['label'] == label
    if not mask.any():
        continue
    spans = regimes.loc[mask, "date"]
    for date in spans:
        ax.axvspan(date - pd.Timedelta(hours=12), date + pd.Timedelta(hours=12), color=label_colors[label], alpha=0.35)

for col in norm.columns:
    ax.plot(norm.index, norm[col], label=col, linewidth=1.6)

ax.set_title('Rule V2 Inputs (Normalized to 100) with Regime Zones')
ax.set_ylabel('Normalized Level')
ax.legend(loc='upper left', ncol=4)

ax2 = axes[1]
ax2.plot(regimes['date'], regimes['score'], label='score', color='black', linewidth=1.8)
ax2.plot(regimes['date'], regimes['prob_risk_on'], label='prob_risk_on', color='#2a7f62', alpha=0.9)
ax2.plot(regimes['date'], regimes['prob_neutral'], label='prob_neutral', color='#ad8b2d', alpha=0.9)
ax2.plot(regimes['date'], regimes['prob_risk_off'], label='prob_risk_off', color='#b33b3b', alpha=0.9)
ax2.axhline(0.15, color='#2a7f62', linestyle='--', linewidth=1.0, alpha=0.7)
ax2.axhline(-0.15, color='#b33b3b', linestyle='--', linewidth=1.0, alpha=0.7)
ax2.set_title('Rule V2 Score and Regime Probabilities')
ax2.set_ylabel('Score / Probability')
ax2.legend(loc='upper left', ncol=4)

plt.tight_layout()
plt.show()


In [ ]:
summary = (
    regimes.groupby('label')
    .agg(days=('date', 'count'), avg_score=('score', 'mean'))
    .sort_values('days', ascending=False)
)
summary
